In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str((Path.cwd().parent / "src").resolve()))

import ee
import geemap
from utils.variables import (
    PROJECT,
    COUNTRIES_ASSET_ID,
    EE_CRS_METERS,
    PSM_CELL_SIZE,
    TREATMENT_CELLS,
    CONTROL_CELLS,
    BIOME_ASSET_ID,
    CALIPER,
    N_NEIGHBORS
)

import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

from absolute_effectiveness.site_selector import SiteSelector
from psm.predict import load_propensity_artifacts, predict_propensity

ee.Authenticate()
ee.Initialize(project=PROJECT)

site_selector = SiteSelector()

import os
os.chdir("/Users/alanalutz/Documents/GitHub/tpae")

EE_CRS_1km = ee.Projection(EE_CRS_METERS).atScale(PSM_CELL_SIZE)

In [ ]:
# Load valid grid cells for a given PA

# Select PA
PA_ID = "1543"
site_id = 1543
test_sites = site_selector.get_test_sites()
site_geom = site_selector.get_site_geom(test_sites, site_id)

# Import treatment and control cells and filter to PA
treatment_cells = gpd.read_parquet(TREATMENT_CELLS).to_crs(epsg=4326)
treatment_cells = treatment_cells[treatment_cells["WDPA_PID"] == PA_ID]
control_cells = gpd.read_parquet(CONTROL_CELLS).to_crs(epsg=4326)
control_cells = control_cells[control_cells["WDPA_PID"] == PA_ID]

# Convert to ee.FeatureCollection
treatment_fc = geemap.geopandas_to_ee(treatment_cells)
control_fc = geemap.geopandas_to_ee(control_cells)
all_cells = ee.FeatureCollection([treatment_fc, control_fc]).flatten()

# Add a unique cell_ID to each grid cell
cell_IDs = ee.List.sequence(0, all_cells.size().getInfo() - 1)
featureList = all_cells.toList(all_cells.size())
grid_fc = ee.FeatureCollection(
    cell_IDs.map(
        lambda cell_ID: ee.Feature(featureList.get(cell_ID)).set(
            {"cell_ID": cell_ID, "label": None}
        )
    )
)

In [ ]:
# Import covariates

elevation_ic = ee.ImageCollection("COPERNICUS/DEM/GLO30").select("DEM")
slope = elevation_ic.map(lambda tile: ee.Terrain.slope(tile)).mosaic().rename("slope")
elevation = elevation_ic.mosaic().rename("elevation")
treecover2000 = ee.Image("UMD/hansen/global_forest_change_2025_v1_13").select("treecover2000")
travel_time = (
    ee.Image("projects/malariaatlasproject/assets/accessibility/accessibility_to_cities/2015_v1_0")
    .select("accessibility").rename("travel_time")
)
log_pop_density = (
    ee.Image("JRC/GHSL/P2023A/GHS_POP/2000")
    .select("population_count")
    .add(1) # handles zeros for log transform
    .log()
    .rename("log_pop_density")
)

# Resample covariates to 1km resolution

def resample(img):
    return (
        img.setDefaultProjection(EE_CRS_1km)
        .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=4096)
        .reproject(EE_CRS_1km)
    )

elevation = resample(elevation)
slope = resample(slope)
treecover2000 = resample(treecover2000)
travel_time = resample(travel_time)
log_pop_density = resample(log_pop_density)

covariates = (
    elevation
    .addBands(slope)
    .addBands(treecover2000)
    .addBands(travel_time)
    .addBands(log_pop_density)
)

In [ ]:
# Aggregate covariates within grid cells
# will need to use mode reducer for categorical covariates

grid_fc = (
    covariates
    .reduceRegions(
        collection=grid_fc,
        reducer=ee.Reducer.mean(),
        scale=PSM_CELL_SIZE,
        crs=EE_CRS_1km,
    )
    .select("cell_ID", "elevation", "slope", "treecover2000", "travel_time", "log_pop_density", "protected")
)

In [ ]:
# Convert cells to centroids
centroids = grid_fc.map(lambda cell: ee.Feature(cell).centroid())

# Assign country and ecoregion to each centroid

countries = ee.FeatureCollection(COUNTRIES_ASSET_ID)
ecoregions = ee.FeatureCollection(BIOME_ASSET_ID)

spatial_filter = ee.Filter.intersects(
    leftField='.geo',
    rightField='.geo',
    maxError=1
)

centroids = ee.Join.saveFirst('_match').apply(
    primary=centroids,
    secondary=countries,
    condition=spatial_filter
).map(lambda f: f
    .set('country', ee.Feature(f.get('_match')).get('country_na'))
    .set('_match', None)
)

centroids = ee.Join.saveFirst('_match').apply(
    primary=centroids,
    secondary=ecoregions,
    condition=spatial_filter
).map(lambda f: f
    .set('ecoregion', ee.Feature(f.get('_match')).get('ECO_ID'))
    .set('biome', ee.Feature(f.get('_match')).get('BIOME_NUM'))
    .set('_match', None)
)

In [ ]:
# Convert centroids to dataframe
cells_list = centroids.getInfo()["features"]
cells_df = pd.DataFrame([feature["properties"] for feature in cells_list])
print(cells_df.head())

In [ ]:
# Load the saved propensity score model from global_psm.ipynb

model_files = sorted(Path("notebooks/models").glob("propensity_model_*.pkl"))
artifacts = load_propensity_artifacts(model_files[-1])
print(f"Loaded {model_files[-1].name}")
print(f"Training AUC: {artifacts['training_metadata']['auc']:.4f}")

# Predict propensity scores for each cell

cells_df["propensity_score"] = predict_propensity(cells_df, artifacts)

print(f"\nCells: {len(cells_df)}")
print(f"Treatment (protected=1): {(cells_df['protected'] == 1).sum()}")
print(f"Control (protected=0): {(cells_df['protected'] == 0).sum()}")
print(f"\nPropensity score distribution:")
print(cells_df["propensity_score"].describe())
print(f"\nFirst 5 rows:")
print(cells_df[["cell_ID", "protected", "biome", "country", "ecoregion", "propensity_score"]].head())

In [ ]:
# Match each treatment cell to up to 4 control cells
# Re-using control cells is ok

# Split treatment and control
treat_df = cells_df[cells_df["protected"] == 1].copy().reset_index(drop=True)
control_df = cells_df[cells_df["protected"] == 0].copy().reset_index(drop=True)

matches = []

# Iterate over (country, ecoregion) strata present in treatment cells.
# Country is a hard constraint; ecoregion relaxes to biome if no within-ecoregion
# controls exist in that country.
for (country, ecoregion), treat_sub in treat_df.groupby(["country", "ecoregion"]):
    control_country = control_df[control_df["country"] == country]

    if len(control_country) == 0:
        print(f"  ({country}, ecoregion {ecoregion}): no controls in country, "
              f"skipping {len(treat_sub)} treatment cells")
        continue

    control_sub = control_country[control_country["ecoregion"] == ecoregion]

    if len(control_sub) == 0:
        # No same-ecoregion controls in this country; fall back to same-biome
        biome = treat_sub["biome"].iloc[0]
        control_sub = control_country[control_country["biome"] == biome]
        fallback = "biome"
        print(f"  ({country}, ecoregion {ecoregion}): no within-ecoregion controls, "
              f"falling back to biome {biome} ({len(control_sub)} controls)")
    else:
        fallback = None

    if len(control_sub) == 0:
        print(f"  ({country}, ecoregion {ecoregion}): no controls at any fallback level, "
              f"skipping {len(treat_sub)} treatment cells")
        continue

    # NN search within this stratum's control pool
    n_neighbors = min(N_NEIGHBORS, len(control_sub))
    nn = NearestNeighbors(n_neighbors=n_neighbors, metric="euclidean")
    nn.fit(control_sub[["propensity_score"]].values)

    distances, indices = nn.kneighbors(treat_sub[["propensity_score"]].values)

    for i, treat_row in enumerate(treat_sub.itertuples()):
        for rank, (dist, j) in enumerate(zip(distances[i], indices[i]), start=1):
            if dist <= CALIPER:
                control_row = control_sub.iloc[j]
                matches.append({
                    "treat_cell_id": treat_row.cell_ID,
                    "control_cell_id": control_row["cell_ID"],
                    "treat_score": treat_row.propensity_score,
                    "control_score": control_row["propensity_score"],
                    "ps_distance": float(dist),
                    "match_rank": rank,
                    "match_country": country,
                    "match_ecoregion": ecoregion,
                    "match_fallback": fallback,
                })

match_df = pd.DataFrame(matches).sort_values("treat_cell_id").reset_index(drop=True)

print(f"\nResults:")
print(f"  Total matched pairs: {len(match_df)}")
print(f"  Unique treatment cells matched: {match_df['treat_cell_id'].nunique()}")
print(f"  Unique control cells used: {match_df['control_cell_id'].nunique()}")

unmatched_treat = set(treat_df["cell_ID"]) - set(match_df["treat_cell_id"])
print(f"  Treatment cells with no match: {len(unmatched_treat)}")

match_coverage = match_df["treat_cell_id"].nunique() / len(treat_df)
print(f"  Match coverage: {match_coverage:.1%}")

if len(match_df) > 0:
    avg_matches = match_df.groupby("treat_cell_id").size().mean()
    print(f"  Avg matches per matched treatment cell: {avg_matches:.2f}")

    control_reuse = match_df.groupby("control_cell_id").size()
    print(f"  Control reuse: min={control_reuse.min()}, "
          f"max={control_reuse.max()}, "
          f"mean={control_reuse.mean():.1f}")

print(f"\nFirst 10 matches:")
print(match_df.head(10) if len(match_df) > 0 else "(no matches)")

In [ ]:
# Filter grid cells to only valid matches

valid_ids = pd.concat([match_df["treat_cell_id"], match_df["control_cell_id"]]).unique()
valid_ids = ee.List(valid_ids.astype(int).tolist())
matched_grids = grid_fc.filter(ee.Filter.inList("cell_ID", valid_ids))

In [ ]:
# Save matched_grids and propensity match pairs to parquet
matched_grids_gdf = geemap.ee_to_gdf(matched_grids)
matched_grids_gdf.to_parquet(f"data/matched_grids_{PA_ID}.parquet")
match_df.to_parquet(f"data/match_table_{PA_ID}.parquet", index=False)